In [ ]:
### streams the full multi-band Sentinel-2, Sentinel-1 SAR and CHIRPS data — the actual inputs to VegFormer. 

In [1]:
import zarr
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('C:/vegformer/data/processed')

# Verify notebook 01 output is intact
z = zarr.open(str(PROCESSED_DIR / 'ndvi_monthly_2019_2024_clean.zarr'), mode='r')
print(f"NDVI array:  {z.shape}  |  NaNs: {np.isnan(z[:]).sum()}  |  Ready: True")
print(f"\nReady to stream:")
print(f"  Sentinel-2  — 14 bands, 16-day composites, 2019-2024")
print(f"  Sentinel-1  — VV + VH SAR, 16-day composites, 2019-2024")
print(f"  CHIRPS      — rainfall + derived climate vars, 2019-2024")

NDVI array:  (72, 377, 313)  |  NaNs: 0  |  Ready: True

Ready to stream:
  Sentinel-2  — 14 bands, 16-day composites, 2019-2024
  Sentinel-1  — VV + VH SAR, 16-day composites, 2019-2024
  CHIRPS      — rainfall + derived climate vars, 2019-2024


In [ ]:
###Perfect. Now set up the full environment for notebook

In [2]:
import ee
import geemap
import numpy as np
import zarr
import requests
import rasterio
from rasterio.io import MemoryFile
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm
import matplotlib.pyplot as plt

ee.Initialize(project='dc-site-scout')

PROCESSED_DIR = Path('C:/vegformer/data/processed')

# AOI — same as notebook 01
aoi_coords = [
    [34.8800, -1.8500], [35.0500, -1.8500], [35.2000, -1.7800],
    [35.3500, -1.6500], [35.4200, -1.5000], [35.4200, -1.1500],
    [35.3000, -1.0500], [35.1500, -1.0000], [34.9500, -1.0000],
    [34.8000, -1.0800], [34.7200, -1.2500], [34.7200, -1.5500],
    [34.7800, -1.7500], [34.8800, -1.8500]
]
aoi = ee.Geometry.Polygon([aoi_coords])

# Streaming function — same as notebook 01
def stream_image_from_gee(image, aoi, scale=250, bands=None):
    if bands:
        image = image.select(bands)
    url = image.getDownloadURL({
        'region': aoi,
        'scale':  scale,
        'crs':    'EPSG:32736',
        'format': 'GEO_TIFF'
    })
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with MemoryFile(response.content) as memfile:
        with memfile.open() as dataset:
            arr = dataset.read().astype(np.float32)
    return arr

# Date windows — 16-day composites 2019-2024
def date_windows(start, end, days=16):
    windows, current = [], datetime.strptime(start, '%Y-%m-%d')
    end_dt = datetime.strptime(end, '%Y-%m-%d')
    while current < end_dt:
        nxt = min(current + timedelta(days=days), end_dt)
        windows.append((current.strftime('%Y-%m-%d'), nxt.strftime('%Y-%m-%d')))
        current = nxt
    return windows

windows = date_windows('2019-01-01', '2024-12-31')
print(f"EE connected")
print(f"16-day windows: {len(windows)}")
print(f"Approx years covered: {len(windows)/23:.1f}")

EE connected
16-day windows: 137
Approx years covered: 6.0


In [ ]:
#####Now stream Sentinel-2. This is the longest cell — it will take 20–30 minutes for all 137 composites. Confirm the test passes first before we run all 137

In [3]:
def mask_s2_clouds(img):
    scl  = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(mask).divide(10000)

def get_s2_composite(start_date, end_date):
    def add_indices(img):
        ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
        evi  = img.expression(
            '2.5*(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)',
            {'NIR':img.select('B8'),'RED':img.select('B4'),'BLUE':img.select('B2')}
        ).rename('EVI')
        nbr  = img.normalizedDifference(['B8','B12']).rename('NBR')
        ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
        return img.addBands([ndvi, evi, nbr, ndwi])

    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
            .select(['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','SCL'])
            .map(mask_s2_clouds)
            .map(add_indices)
            .select(['B2','B3','B4','B5','B6','B7',
                     'B8','B8A','B11','B12','NDVI','EVI','NBR','NDWI'])
            .median()
            .clip(aoi))

S2_BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12',
            'NDVI','EVI','NBR','NDWI']

# Test one composite first
print("Testing single S2 composite...")
test = get_s2_composite('2021-04-01', '2021-04-16')
arr  = stream_image_from_gee(test, aoi, scale=250, bands=S2_BANDS)
print(f"Single composite shape: {arr.shape}")  # should be (14, 377, 313)
print(f"Memory: {arr.nbytes/1e6:.1f} MB")
print(f"NDVI band min/max: {arr[10].min():.3f} / {arr[10].max():.3f}")
print("Test passed — starting full stream...")

Testing single S2 composite...
Single composite shape: (14, 377, 313)
Memory: 6.6 MB
NDVI band min/max: -0.059 / 0.916
Test passed — starting full stream...


In [ ]:
######Now run the full 137 composites. This will take 20–30 minutes — let it run

In [4]:
s2_stack = []
s2_dates = []
s2_failed = []

print(f"Streaming {len(windows)} Sentinel-2 composites (20-30 mins)...")
for i, (start, end) in enumerate(tqdm(windows, desc='S2')):
    img = get_s2_composite(start, end)
    try:
        arr = stream_image_from_gee(img, aoi, scale=250, bands=S2_BANDS)
        s2_stack.append(arr)
        s2_dates.append(start)
    except Exception as e:
        # Fill with NaN array on failure
        if s2_stack:
            s2_stack.append(np.full_like(s2_stack[-1], np.nan))
        else:
            s2_stack.append(np.full((14, 377, 313), np.nan))
        s2_dates.append(start)
        s2_failed.append((start, str(e)))

s2_array = np.stack(s2_stack, axis=0)  # (137, 14, 377, 313)

print(f"\nDone.")
print(f"Shape:          {s2_array.shape}")
print(f"Memory:         {s2_array.nbytes/1e6:.1f} MB")
print(f"Failed windows: {len(s2_failed)}")
if s2_failed:
    for date, err in s2_failed:
        print(f"  {date}: {err}")

Streaming 137 Sentinel-2 composites (20-30 mins)...


S2: 100%|████████████████████████████████████████████████████████████████████████████| 137/137 [34:51<00:00, 15.27s/it]



Done.
Shape:          (137, 14, 377, 313)
Memory:         905.3 MB
Failed windows: 1
  2022-11-25: 503 Server Error: Service Unavailable for url: https://earthengine.googleapis.com/v1/projects/dc-site-scout/thumbnails/1c1f323979447650c007980b888299d1-313dd0e9f7500dbd4027d2112a429014:getPixels


In [5]:
# Save immediately — don't run any other cells first
print("Saving S2 array to disk now...")

z_s2 = zarr.open(
    str(PROCESSED_DIR / 's2_16day_2019_2024.zarr'),
    mode='w',
    shape=s2_array.shape,
    dtype='float32',
    chunks=(1, 14, 377, 313),
)
z_s2[:] = s2_array
z_s2.attrs['dates']     = s2_dates
z_s2.attrs['bands']     = S2_BANDS
z_s2.attrs['aoi']       = 'Maasai Mara Ecosystem'
z_s2.attrs['scale']     = '250m'
z_s2.attrs['crs']       = 'EPSG:32736'
z_s2.attrs['completed'] = len(s2_dates)
z_s2.attrs['failed']    = ['2022-11-25']

size = sum(f.stat().st_size for f in
           (PROCESSED_DIR / 's2_16day_2019_2024.zarr').rglob('*'))
print(f"Saved: s2_16day_2019_2024.zarr")
print(f"Shape: {z_s2.shape}")
print(f"Size on disk: {size/1e6:.1f} MB")
print(f"Composites saved: {len(s2_dates)}")

Saving S2 array to disk now...
Saved: s2_16day_2019_2024.zarr
Shape: (137, 14, 377, 313)
Size on disk: 614.5 MB
Composites saved: 137


In [6]:
# Fix the one failed composite (2022-11-25)
failed_idx = s2_dates.index('2022-11-25') if '2022-11-25' in s2_dates else None

if failed_idx is not None:
    print(f"Fixing failed composite at index {failed_idx} (2022-11-25)...")
    before = s2_array[failed_idx - 1]
    after  = s2_array[failed_idx + 1]
    s2_array[failed_idx] = (before + after) / 2
    
    # Check it's no longer NaN
    nan_count = np.isnan(s2_array[failed_idx]).sum()
    print(f"NaN values after fix: {nan_count}")
    print(f"Fixed composite mean NDVI: {s2_array[failed_idx, 10].mean():.3f}")

    # Overwrite the saved zarr with the fix
    z_s2 = zarr.open(str(PROCESSED_DIR / 's2_16day_2019_2024.zarr'), mode='r+')
    z_s2[failed_idx] = s2_array[failed_idx]
    z_s2.attrs['failed'] = []
    z_s2.attrs['interpolated'] = ['2022-11-25']
    print("Fix saved to disk")
else:
    print("No failed composites found in dates list")

# Final quality check
print(f"\nFinal S2 array quality:")
print(f"  Shape:          {s2_array.shape}")
print(f"  NaN values:     {np.isnan(s2_array).sum()}")
print(f"  Min NDVI:       {s2_array[:, 10].min():.3f}")
print(f"  Max NDVI:       {s2_array[:, 10].max():.3f}")
print(f"  Mean NDVI:      {s2_array[:, 10].mean():.3f}")
print(f"  Memory:         {s2_array.nbytes/1e6:.1f} MB")
print(f"  Size on disk:   614.5 MB")

Fixing failed composite at index 89 (2022-11-25)...
NaN values after fix: 0
Fixed composite mean NDVI: 0.306
Fix saved to disk

Final S2 array quality:
  Shape:          (137, 14, 377, 313)
  NaN values:     0
  Min NDVI:       -0.427
  Max NDVI:       1.000
  Mean NDVI:      0.513
  Memory:         905.3 MB
  Size on disk:   614.5 MB


In [ ]:
##### stream Sentinel-1 SAR — much faster than S2 since it's only 2 bands:


In [7]:
def get_s1_composite(start_date, end_date):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
            .select(['VV','VH'])
            .map(lambda img: img.focalMedian(3))
            .median()
            .clip(aoi))

S1_BANDS = ['VV', 'VH']

# Test one composite first
print("Testing single S1 composite...")
test = get_s1_composite('2021-04-01', '2021-04-16')
arr  = stream_image_from_gee(test, aoi, scale=250, bands=S1_BANDS)
print(f"Shape:  {arr.shape}")       # should be (2, 377, 313)
print(f"Memory: {arr.nbytes/1e6:.1f} MB")
print(f"VV range: {arr[0].min():.2f} to {arr[0].max():.2f} dB")
print(f"VH range: {arr[1].min():.2f} to {arr[1].max():.2f} dB")
print("Test passed — starting full stream...")

# Full stream — saves every 20 composites to protect against crashes
s1_stack  = []
s1_dates  = []
s1_failed = []

print(f"\nStreaming {len(windows)} S1 composites (~15 mins)...")
for i, (start, end) in enumerate(tqdm(windows, desc='S1')):
    img = get_s1_composite(start, end)
    try:
        arr = stream_image_from_gee(img, aoi, scale=250, bands=S1_BANDS)
        s1_stack.append(arr)
        s1_dates.append(start)
    except Exception as e:
        if s1_stack:
            s1_stack.append(np.full_like(s1_stack[-1], np.nan))
        else:
            s1_stack.append(np.full((2, 377, 313), np.nan))
        s1_dates.append(start)
        s1_failed.append((start, str(e)))

    # Save checkpoint every 20 composites
    if (i + 1) % 20 == 0:
        checkpoint = np.stack(s1_stack, axis=0)
        np.save(str(PROCESSED_DIR / 's1_checkpoint.npy'), checkpoint)
        print(f"  Checkpoint saved at {i+1}/{len(windows)}")

s1_array = np.stack(s1_stack, axis=0)  # (137, 2, 377, 313)

print(f"\nDone.")
print(f"Shape:          {s1_array.shape}")
print(f"Memory:         {s1_array.nbytes/1e6:.1f} MB")
print(f"Failed windows: {len(s1_failed)}")
if s1_failed:
    for date, err in s1_failed[:3]:
        print(f"  {date}: {err}")

Testing single S1 composite...
Shape:  (2, 377, 313)
Memory: 0.9 MB
VV range: -16.23 to -6.46 dB
VH range: -22.59 to -13.60 dB
Test passed — starting full stream...

Streaming 137 S1 composites (~15 mins)...


S1:  15%|███████████▏                                                                 | 20/137 [01:10<07:34,  3.89s/it]

  Checkpoint saved at 20/137


S1:  29%|██████████████████████▍                                                      | 40/137 [02:26<05:57,  3.69s/it]

  Checkpoint saved at 40/137


S1:  44%|█████████████████████████████████▋                                           | 60/137 [03:39<04:10,  3.25s/it]

... (5 outputs left out)

S1:  88%|██████████████████████████████████████████████████████████████████▌         | 120/137 [04:38<00:15,  1.11it/s]

  Checkpoint saved at 120/137


S1: 100%|████████████████████████████████████████████████████████████████████████████| 137/137 [04:46<00:00,  2.09s/it]


Done.
Shape:          (137, 2, 377, 313)
Memory:         258.7 MB
Failed windows: 75
  2019-01-01: Image.select: Band pattern 'VV' was applied to an Image with no bands. See https://developers.google.com/earth-engine/guides/debugging#no-bands
  2019-04-07: Image.select: Band pattern 'VV' was applied to an Image with no bands. See https://developers.google.com/earth-engine/guides/debugging#no-bands
  2019-06-26: Image.select: Band pattern 'VV' was applied to an Image with no bands. See https://developers.google.com/earth-engine/guides/debugging#no-bands


In [8]:
def get_s1_composite_both_orbits(start_date, end_date):
    """Use both ascending and descending passes to maximize coverage."""
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .select(['VV','VH'])
            .map(lambda img: img.focalMedian(3))
            .median()
            .clip(aoi))

# Test on a previously failed date
print("Testing previously failed date (2019-01-01)...")
test = get_s1_composite_both_orbits('2019-01-01', '2019-01-17')
try:
    arr = stream_image_from_gee(test, aoi, scale=250, bands=['VV','VH'])
    print(f"Shape:    {arr.shape}")
    print(f"VV range: {arr[0].min():.2f} to {arr[0].max():.2f} dB")
    print("Fix works — restreaming all 75 failed windows...")
except Exception as e:
    print(f"Still failing: {e}")

Testing previously failed date (2019-01-01)...
Shape:    (2, 377, 313)
VV range: -27.07 to -5.90 dB
Fix works — restreaming all 75 failed windows...


In [9]:
# Find which indices failed — these have all NaN values
failed_indices = [i for i in range(len(s1_stack)) 
                  if np.isnan(s1_array[i]).all()]
print(f"Restreaming {len(failed_indices)} failed composites...")

fixed = 0
still_failed = []

for i in tqdm(failed_indices, desc='Fixing S1'):
    start, end = windows[i]
    img = get_s1_composite_both_orbits(start, end)
    try:
        arr = stream_image_from_gee(img, aoi, scale=250, bands=['VV','VH'])
        s1_array[i] = arr
        fixed += 1
    except Exception as e:
        still_failed.append((start, str(e)))

print(f"\nFixed:        {fixed}/{len(failed_indices)}")
print(f"Still failed: {len(still_failed)}")

# Interpolate any remaining failures
if still_failed:
    print("Interpolating remaining failures...")
    for i in range(s1_array.shape[0]):
        if np.isnan(s1_array[i]).all():
            before = next((j for j in range(i-1,-1,-1) 
                          if not np.isnan(s1_array[j]).all()), None)
            after  = next((j for j in range(i+1, s1_array.shape[0]) 
                          if not np.isnan(s1_array[j]).all()), None)
            if before is not None and after is not None:
                s1_array[i] = (s1_array[before] + s1_array[after]) / 2
                print(f"  Interpolated index {i} ({windows[i][0]})")

# Final check
print(f"\nFinal S1 quality:")
print(f"  Shape:      {s1_array.shape}")
print(f"  NaN values: {np.isnan(s1_array).sum()}")
print(f"  VV range:   {np.nanmin(s1_array[:,0]):.2f} to {np.nanmax(s1_array[:,0]):.2f} dB")
print(f"  VH range:   {np.nanmin(s1_array[:,1]):.2f} to {np.nanmax(s1_array[:,1]):.2f} dB")

Restreaming 75 failed composites...


Fixing S1: 100%|███████████████████████████████████████████████████████████████████████| 75/75 [05:11<00:00,  4.16s/it]



Fixed:        75/75
Still failed: 0

Final S1 quality:
  Shape:      (137, 2, 377, 313)
  NaN values: 0
  VV range:   -33.06 to 0.00 dB
  VH range:   -46.22 to 0.00 dB


In [10]:
# Replace exact 0.0 dB values — specular reflection artifact
# Valid SAR backscatter range for vegetation: -25 to -3 dB
s1_array[s1_array == 0.0] = np.nan

# Interpolate the zeros per pixel across time
from scipy.interpolate import interp1d

def interpolate_sar_zeros(array):
    """Per-pixel interpolation for SAR artifact zeros."""
    T, B, H, W = array.shape
    filled = array.copy()
    fixed  = 0
    for b in range(B):
        for r in range(H):
            for c in range(W):
                pixel    = array[:, b, r, c]
                nan_mask = np.isnan(pixel)
                valid    = ~nan_mask
                if nan_mask.any() and valid.sum() >= 2:
                    f = interp1d(
                        np.where(valid)[0], pixel[valid],
                        kind='linear', bounds_error=False,
                        fill_value=(pixel[valid][0], pixel[valid][-1])
                    )
                    filled[nan_mask, b, r, c] = f(np.where(nan_mask)[0])
                    fixed += nan_mask.sum()
    print(f"  Fixed {fixed:,} artifact values")
    return filled

print("Fixing SAR specular artifacts...")
print("(Takes 3-4 minutes)")
s1_array = interpolate_sar_zeros(s1_array)

print(f"\nFinal S1 quality:")
print(f"  NaN values: {np.isnan(s1_array).sum()}")
print(f"  VV range:   {s1_array[:,0].min():.2f} to {s1_array[:,0].max():.2f} dB")
print(f"  VH range:   {s1_array[:,1].min():.2f} to {s1_array[:,1].max():.2f} dB")

# Save
z_s1 = zarr.open(
    str(PROCESSED_DIR / 's1_16day_2019_2024.zarr'),
    mode='w',
    shape=s1_array.shape,
    dtype='float32',
    chunks=(1, 2, 377, 313),
)
z_s1[:] = s1_array
z_s1.attrs['dates']  = s1_dates
z_s1.attrs['bands']  = ['VV', 'VH']
z_s1.attrs['aoi']    = 'Maasai Mara Ecosystem'
z_s1.attrs['scale']  = '250m'
z_s1.attrs['crs']    = 'EPSG:32736'
z_s1.attrs['units']  = 'dB'

size = sum(f.stat().st_size for f in
           (PROCESSED_DIR / 's1_16day_2019_2024.zarr').rglob('*'))
print(f"\nSaved: s1_16day_2019_2024.zarr")
print(f"Size on disk: {size/1e6:.1f} MB")

Fixing SAR specular artifacts...
(Takes 3-4 minutes)
  Fixed 310,848 artifact values

Final S1 quality:
  NaN values: 0
  VV range:   -33.06 to -4.23 dB
  VH range:   -46.22 to -10.48 dB

Saved: s1_16day_2019_2024.zarr
Size on disk: 85.7 MB


In [11]:
def get_chirps_composite(start_date, end_date):
    """16-day rainfall sum + derived climate variables."""
    chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
              .filterDate(start_date, end_date)
              .filterBounds(aoi))

    rain_sum  = chirps.sum().rename('rain_sum')
    dry_days  = chirps.map(lambda img: img.lt(1)).sum().rename('dry_days')
    rain_mean = chirps.mean().rename('rain_mean')

    # 90-day anomaly vs long-term mean
    start_90  = ee.Date(start_date).advance(-90, 'day')
    rain_90   = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                 .filterDate(start_90, end_date)
                 .filterBounds(aoi)
                 .sum().rename('rain_90d'))

    return (rain_sum.addBands([dry_days, rain_mean, rain_90])
                    .clip(aoi))

CHIRPS_BANDS = ['rain_sum', 'dry_days', 'rain_mean', 'rain_90d']

# Test one composite
print("Testing single CHIRPS composite...")
test = get_chirps_composite('2021-04-01', '2021-04-16')
arr  = stream_image_from_gee(test, aoi, scale=5000, bands=CHIRPS_BANDS)
print(f"Shape:         {arr.shape}")
print(f"Rain sum range: {arr[0].min():.1f} to {arr[0].max():.1f} mm")
print(f"Dry days range: {arr[1].min():.0f} to {arr[1].max():.0f} days")
print("Test passed — starting full stream...")

# Full stream
chirps_stack  = []
chirps_dates  = []
chirps_failed = []

print(f"\nStreaming {len(windows)} CHIRPS composites (~5 mins)...")
for i, (start, end) in enumerate(tqdm(windows, desc='CHIRPS')):
    img = get_chirps_composite(start, end)
    try:
        arr = stream_image_from_gee(img, aoi, scale=5000, bands=CHIRPS_BANDS)
        chirps_stack.append(arr)
        chirps_dates.append(start)
    except Exception as e:
        if chirps_stack:
            chirps_stack.append(np.full_like(chirps_stack[-1], np.nan))
        else:
            chirps_stack.append(np.full((4, 377, 313), np.nan))
        chirps_dates.append(start)
        chirps_failed.append((start, str(e)))

    # Checkpoint every 30 composites
    if (i + 1) % 30 == 0:
        np.save(str(PROCESSED_DIR / 'chirps_checkpoint.npy'),
                np.stack(chirps_stack, axis=0))
        print(f"  Checkpoint saved at {i+1}/{len(windows)}")

chirps_array = np.stack(chirps_stack, axis=0)  # (137, 4, H, W)

print(f"\nDone.")
print(f"Shape:          {chirps_array.shape}")
print(f"Memory:         {chirps_array.nbytes/1e6:.1f} MB")
print(f"Failed windows: {len(chirps_failed)}")

Testing single CHIRPS composite...
Shape:         (4, 19, 16)
Rain sum range: 71.8 to 150.5 mm
Dry days range: 2 to 12 days
Test passed — starting full stream...

Streaming 137 CHIRPS composites (~5 mins)...


CHIRPS:  22%|███████████████▉                                                         | 30/137 [01:09<05:21,  3.01s/it]

  Checkpoint saved at 30/137


CHIRPS:  44%|███████████████████████████████▉                                         | 60/137 [02:15<02:47,  2.18s/it]

  Checkpoint saved at 60/137


CHIRPS:  66%|███████████████████████████████████████████████▉                         | 90/137 [03:26<01:33,  1.99s/it]

  Checkpoint saved at 90/137


CHIRPS:  88%|███████████████████████████████████████████████████████████████         | 120/137 [04:34<00:36,  2.15s/it]

  Checkpoint saved at 120/137


CHIRPS: 100%|████████████████████████████████████████████████████████████████████████| 137/137 [05:07<00:00,  2.24s/it]


Done.
Shape:          (137, 4, 19, 16)
Memory:         0.7 MB
Failed windows: 0


In [12]:
# Save CHIRPS
z_chirps = zarr.open(
    str(PROCESSED_DIR / 'chirps_16day_2019_2024.zarr'),
    mode='w',
    shape=chirps_array.shape,
    dtype='float32',
    chunks=(1, 4, 19, 16),
)
z_chirps[:] = chirps_array
z_chirps.attrs['dates'] = chirps_dates
z_chirps.attrs['bands'] = CHIRPS_BANDS
z_chirps.attrs['aoi']   = 'Maasai Mara Ecosystem'
z_chirps.attrs['scale'] = '5000m'
z_chirps.attrs['crs']   = 'EPSG:32736'
z_chirps.attrs['units'] = 'mm'

size = sum(f.stat().st_size for f in
           (PROCESSED_DIR / 'chirps_16day_2019_2024.zarr').rglob('*'))
print(f"Saved: chirps_16day_2019_2024.zarr | {size/1e6:.1f} MB")

# Clean up checkpoints
import os
for f in ['s1_checkpoint.npy', 'chirps_checkpoint.npy']:
    p = PROCESSED_DIR / f
    if p.exists():
        os.remove(p)
        print(f"Removed checkpoint: {f}")

# Final summary of all processed data
print("\n" + "="*55)
print("  Notebook 02 complete — all sensors streamed")
print("="*55)

datasets = [
    ('ndvi_monthly_2019_2024_clean.zarr', 'NDVI monthly'),
    ('s2_16day_2019_2024.zarr',           'Sentinel-2'),
    ('s1_16day_2019_2024.zarr',           'Sentinel-1'),
    ('chirps_16day_2019_2024.zarr',       'CHIRPS'),
]

total_mb = 0
for fname, name in datasets:
    path = PROCESSED_DIR / fname
    z    = zarr.open(str(path), mode='r')
    mb   = sum(f.stat().st_size for f in path.rglob('*')) / 1e6
    total_mb += mb
    print(f"  {name:<20} shape={str(z.shape):<25} "
          f"NaNs={np.isnan(z[:]).sum():<8} {mb:.1f} MB")

print(f"\n  Total on disk: {total_mb:.1f} MB")
print(f"  Location: {PROCESSED_DIR}")
print("="*55)
print("\nReady for Notebook 03 — patch extraction + VegFormer training")

Saved: chirps_16day_2019_2024.zarr | 0.4 MB
Removed checkpoint: s1_checkpoint.npy
Removed checkpoint: chirps_checkpoint.npy

  Notebook 02 complete — all sensors streamed
  NDVI monthly         shape=(72, 377, 313)            NaNs=0        29.1 MB
  Sentinel-2           shape=(137, 14, 377, 313)       NaNs=0        619.4 MB
  Sentinel-1           shape=(137, 2, 377, 313)        NaNs=0        85.7 MB
  CHIRPS               shape=(137, 4, 19, 16)          NaNs=0        0.4 MB

  Total on disk: 734.8 MB
  Location: C:\vegformer\data\processed

Ready for Notebook 03 — patch extraction + VegFormer training
